# Table 1 and numbers quoted in the text

This notebook prints two blocks of LaTeX to **copy into `paper/main.tex`**.
Nothing is written to a separate `.tex` file, because many journals reject
submissions that `\\input` other files.

| Block | Where it goes in `main.tex` |
|---|---|
| Table 1 `tabular` | Inside the `table` environment, replacing the old `tabular` |
| `\\newcommand` definitions | The preamble, replacing the old definitions |

Below each block the notebook reports whether `main.tex` already contains
it exactly, so after the data changes you can see at a glance what needs
pasting. Defining the numbers as macros means each value is pasted once,
in the preamble, and every mention in the text updates with it.

In [ ]:
import sys
from pathlib import Path

# Make the repository's shared helpers importable from this notebook
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from common import paths

paths.use_paper_style()
data = np.genfromtxt(paths.DATA_DIR / "example_measurements.csv",
                     delimiter=",", names=True)
group_names = "ABCD"
groups = np.unique(data["group"]).astype(int)

In [ ]:
def fit_line(x, y, y_err):
    """Weighted least-squares straight-line fit.

    Returns (slope, intercept), their 1-sigma uncertainties and the
    chi-squared per degree of freedom.
    """
    coeffs, cov = np.polyfit(x, y, 1, w=1 / y_err, cov="unscaled")
    resid = (y - np.polyval(coeffs, x)) / y_err
    chi2_dof = np.sum(resid**2) / (x.size - 2)
    return coeffs, np.sqrt(np.diag(cov)), chi2_dof

fits = {}
for g in groups:
    sel = data["group"] == g
    fits[g] = fit_line(data["x"][sel], data["y"][sel], data["y_err"][sel])

## Table 1: fit results

In [ ]:
lines = [
    r"\begin{tabular}{c|cc|c}",
    r"\hline",
    r"Group & Slope & Intercept & $\chi^2/\mathrm{dof}$ \\",
    r"\hline",
]
for g in groups:
    (m, c), (m_err, c_err), chi2_dof = fits[g]
    lines.append(f"{group_names[g]} & ${m:.2f} \\pm {m_err:.2f}$ & "
                 f"${c:.2f} \\pm {c_err:.2f}$ & {chi2_dof:.2f} \\\\")
lines += [r"\hline", r"\end{tabular}"]
table_tex = "\n".join(lines) + "\n"

paths.show_for_paper(table_tex, "the table environment for Table 1")

## Numbers quoted in the text

Each value becomes a `\newcommand` so the paper text can say *"the steepest slope, \SteepestSlope, is found for group \SteepestGroup"*.

In [ ]:
slopes = {g: fits[g][0][0] for g in groups}
steepest = max(slopes, key=slopes.get)
macros = {
    "NGroups": len(groups),
    "NPointsPerGroup": int(np.sum(data["group"] == groups[0])),
    "SteepestGroup": group_names[steepest],
    "SteepestSlope": f"${slopes[steepest]:.2f} \\pm {fits[steepest][1][0]:.2f}$",
    "MaxChiSqPerDof": f"{max(fits[g][2] for g in groups):.2f}",
}
macro_tex = "% Generated by analysis/table_1_fit_results.ipynb -- do not edit by hand\n"
macro_tex += "".join(f"\\newcommand{{\\{k}}}{{{v}}}\n" for k, v in macros.items())

paths.show_for_paper(macro_tex, "the preamble")

The same values can be shown on this page with MyST's `glue` if you want the
web page and paper to quote identical numbers, e.g.
```python
from myst_nb import glue
glue("steepest_group", macros["SteepestGroup"])
```
and then write `` {glue:}`steepest_group` `` in any Markdown page.